In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28,28)), # TODO: Resize to 28x28
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    transforms.ToTensor(),    # TODO: Convert to Tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225]) # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

data_iter = iter(train_loader)
images, _ = next(data_iter)
# Show images
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
  img = images[i]
  img = np.transpose(img.numpy(), (1, 2, 0))
  ax.imshow(img)
  ax.axis("off")
plt.show()
print("Shape of one image tensor:", images[0].shape)

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
class efficientnet_v2_s(nn.Module):
  def __init__(self, num_classes):
    super(efficientnet_v2_s, self).__init__()
    self.features = nn.Sequential(
      nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),
      nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3),
      nn.ReLU()
)
    self.classifier = nn.Sequential(
      nn.Linear(121, 32),
      nn.Linear(32, 32)
      )
  def forward(self, x):
    x = self.features(x)
    x = torch.flatten(x, 1)
    x = self.classifier(x)
    return x

In [ ]:
# Write your code here
from tqdm import tqdm
def train_one_epoch(model, dataloader, criterion, optimizer, device):
  model.train()
  print(device)
  total_loss = 0
  correct = 0
  total = 0
  for images, labels in tqdm(dataloader):
    images, labels = images.to(device), labels
    outputs = model(images)
    loss = criterion(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
    outputs = torch.softmax(outputs, dim=1)
    predictions = outputs.argmax(dim=1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)
  avg_loss = total_loss / len(dataloader)
  accuracy = 100 * correct / total
  return avg_loss, accuracy




In [ ]:
# Write your code here
from tqdm import tqdm
def  validate(model, dataloader, criterion, device):
  model.eval()
  print(device)
  total_loss = 0
  correct = 0
  total = 0
  with torch.no_grad():
    for images, labels in tqdm(dataloader):
      images, labels = images.to(device), labels
      outputs = model(images)
      loss = criterion(outputs, labels)
      total_loss += loss.item()
      outputs = torch.softmax(outputs, dim=1)
      predictions = outputs.argmax(dim=1)
      correct += (predictions == labels).sum().item()
      total += labels.size(0)
  avg_loss = total_loss / len(dataloader)
  accuracy = 100 * correct / total
  return avg_loss, accuracy


In [ ]:
# Write your code here
import torch.optim as optim
dataloader= train_dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = efficientnet_v2_s(num_classes=26).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []
for epoch in range(num_epochs):
  train_loss, train_accuracy = train_one_epoch(model, dataloader, criterion, optimizer, device)
  val_loss, val_accuracy = validate(model, dataloader, criterion, device)
  train_losses.append(train_loss)
  val_losses.append(val_loss)
  train_accuracies.append(train_accuracy)
  val_accuracies.append(val_accuracy)
  print(f"Epoch {epoch+1}/{num_epochs}: "
  f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2}"
  f"Val Loss={val_loss:.4f}, Val_Accuracy={val_accuracy:.2f}%")


In [ ]:
# Write your code here
